# dim_date_scd0 (SCD 0, poc method)

Calendar 2005–2030, one row per day, `date_key = yyyymmdd`, `-1` = Unknown. Generated (no source table). `is_vn_holiday` / `holiday_name` from the seed below (fixed-date holidays + Tết and Giỗ Tổ Hùng Vương as solar dates). Same content as `dim_date`.

`full`: overwrite. `incremental`: MERGE on `primary_checksum`, so a run writes only days whose content changed (e.g. a holiday added to the seed).

In [ ]:
%run ./_scd

# Set up environment

In [ ]:
dbutils.widgets.text("gold", "prod.gold")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])

gold = dbutils.widgets.get("gold")
load_type = dbutils.widgets.get("load_type")
path = f"{gold}.dim_date_scd0"
if not spark.catalog.tableExists(path):
    load_type = "full"
print(f"load_type = {load_type}")

# Read Source: VN holiday seed (lunar holidays as solar dates)

In [ ]:
FIXED_HOLIDAYS = {"01-01": "Tết Dương lịch", "04-30": "Ngày Giải phóng miền Nam", "05-01": "Quốc tế Lao động", "09-02": "Quốc khánh"}
TET = ["2020-01-25", "2021-02-12", "2022-02-01", "2023-01-22", "2024-02-10", "2025-01-29",
       "2026-02-17", "2027-02-06", "2028-01-26", "2029-02-13", "2030-02-03"]
HUNG_KINGS = ["2020-04-02", "2021-04-21", "2022-04-10", "2023-04-29", "2024-04-18", "2025-04-07",
              "2026-04-26", "2027-04-16", "2028-04-04", "2029-04-23", "2030-04-12"]

seed_rows = [(f"{y}-{md}", name) for y in range(2005, 2031) for md, name in FIXED_HOLIDAYS.items()]
seed_rows += [(d, "Tết Nguyên Đán") for d in TET] + [(d, "Giỗ Tổ Hùng Vương") for d in HUNG_KINGS]
spark.createDataFrame(seed_rows, "holiday_date STRING, holiday_name STRING").createOrReplaceTempView("vn_holidays")

# Transformation

In [ ]:
days = spark.sql("""
    WITH days AS (
        SELECT explode(sequence(DATE'2005-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS full_date
    )
    SELECT
        CAST(date_format(d.full_date, 'yyyyMMdd') AS INT)      AS date_key,
        d.full_date,
        CAST(day(d.full_date) AS TINYINT)                       AS day_of_month,
        CAST(weekday(d.full_date) + 1 AS TINYINT)               AS day_of_week,
        date_format(d.full_date, 'EEEE')                        AS day_name,
        weekday(d.full_date) + 1 IN (6, 7)                      AS is_weekend,
        CAST(weekofyear(d.full_date) AS TINYINT)                AS week_of_year,
        CAST(month(d.full_date) AS TINYINT)                     AS month,
        date_format(d.full_date, 'MMMM')                        AS month_name,
        CAST(quarter(d.full_date) AS TINYINT)                   AS quarter,
        CAST(year(d.full_date) AS SMALLINT)                     AS year,
        date_format(d.full_date, 'yyyy-MM')                     AS year_month,
        h.holiday_name IS NOT NULL                              AS is_vn_holiday,
        h.holiday_name
    FROM days d
    LEFT JOIN (
        SELECT CAST(holiday_date AS DATE) AS holiday_date, concat_ws(' / ', collect_list(holiday_name)) AS holiday_name
        FROM vn_holidays GROUP BY 1
    ) h ON h.holiday_date = d.full_date
""")

result_df = days.select(
    F.col("date_key").cast("int").alias("date_key"),
    F.col("full_date").cast("date").alias("full_date"),
    F.col("day_of_month").cast("tinyint").alias("day_of_month"),
    F.col("day_of_week").cast("tinyint").alias("day_of_week"),
    F.col("day_name").cast("string").alias("day_name"),
    F.col("is_weekend").cast("boolean").alias("is_weekend"),
    F.col("week_of_year").cast("tinyint").alias("week_of_year"),
    F.col("month").cast("tinyint").alias("month"),
    F.col("month_name").cast("string").alias("month_name"),
    F.col("quarter").cast("tinyint").alias("quarter"),
    F.col("year").cast("smallint").alias("year"),
    F.col("year_month").cast("string").alias("year_month"),
    F.col("is_vn_holiday").cast("boolean").alias("is_vn_holiday"),
    F.col("holiday_name").cast("string").alias("holiday_name"),
)

## Unknown row (date_key = -1)

In [ ]:
unknown = spark.sql("""
    SELECT -1 AS date_key, NULL AS full_date, NULL AS day_of_month, NULL AS day_of_week, 'Unknown' AS day_name,
           NULL AS is_weekend, NULL AS week_of_year, NULL AS month, 'Unknown' AS month_name, NULL AS quarter,
           NULL AS year, 'Unknown' AS year_month, false AS is_vn_holiday, NULL AS holiday_name
""").select(*[F.col(f.name).cast(f.dataType).alias(f.name) for f in result_df.schema])
result_df = result_df.unionByName(unknown)

## Add metadata

In [ ]:
insert_date = datetime.now()
etl_by = "Databricks"
result_df = result_df.withColumn("etl_date", F.lit(insert_date)).withColumn("etl_by", F.lit(etl_by))

## Add checksums

In [ ]:
primary_key = ["date_key"]
result_df = add_checksums(result_df, primary_key)

## Create table

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {path} (
        date_key INT, full_date DATE, day_of_month TINYINT, day_of_week TINYINT, day_name STRING, is_weekend BOOLEAN,
        week_of_year TINYINT, month TINYINT, month_name STRING, quarter TINYINT, year SMALLINT, year_month STRING,
        is_vn_holiday BOOLEAN, holiday_name STRING,
        etl_date TIMESTAMP, etl_by STRING, primary_checksum STRING, record_checksum STRING
    ) USING DELTA
    COMMENT 'Calendar dimension 2005-2030, SCD 0 (poc method). date_key = yyyymmdd, -1 = Unknown.'
""")

# Write to Table

In [ ]:
if load_type == "full":
    full_refresh(result_df, path)
else:
    result_df.createOrReplaceTempView("source_df")
    spark.sql(f"""
        MERGE INTO {path} AS t
        USING source_df AS s
        ON t.primary_checksum = s.primary_checksum
        WHEN MATCHED AND t.record_checksum <> s.record_checksum THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

## Check: one row per day

In [ ]:
c = spark.sql(f"SELECT COUNT(*) AS n, COUNT(DISTINCT date_key) AS keys FROM {path}").first()
assert c["n"] == c["keys"], f"{path}: duplicate date_key"
print(f"✅ {path}: {c['n']} days")
last = spark.sql(f"DESCRIBE HISTORY {path} LIMIT 1").first()
print(last["operation"], dict(last["operationMetrics"] or {}))